In [ ]:
"""
KAP v1.2 — Gemini Edition (with Rate Limiting)
===============================================
Paste this ENTIRE thing into ONE cell in Kaggle.
"""

import os, sys, json, traceback, re, io, time
from datetime import datetime
from contextlib import redirect_stdout, redirect_stderr

# ── CONFIG ──
KAP_VERSION = "1.2-gemini"
STATE_FILE = "/kaggle/working/kap_state.json"
LOG_FILE = "/kaggle/working/kap_log.json"
MAX_RETRIES = 2
MAX_CODE_CHARS = 40000
MAX_ERR_CHARS = 8000
LLM_COOLDOWN_SEC = 6  # Wait between LLM calls (stays under 15/min)

# Kaggle Secrets — GEMINI API KEY
try:
    from kaggle_secrets import UserSecretsClient
    _usc = UserSecretsClient()
    GEMINI_API_KEY = _usc.get_secret("GEMINI_API_KEY") or os.environ.get("GEMINI_API_KEY", "")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL = "gemini-3.1-flash-lite"

if not GEMINI_API_KEY:
    print("⚠️  KAP WARNING: No GEMINI_API_KEY found. Auto-debug is DISABLED.")
else:
    print(f"✅ GEMINI_API_KEY loaded ({len(GEMINI_API_KEY)} chars)")

# ── STATE ──
_kap_state = {}
_kap_log = {"sessions": []}
_kap_runner = None
_last_llm_call = 0

def _now(): return datetime.now().isoformat()
def _ts(): return datetime.now().strftime("%Y%m%d_%H%M%S")

def _load_state():
    global _kap_state
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r") as f:
                _kap_state = json.load(f)
        except Exception:
            pass
    if not _kap_state:
        _kap_state = {
            "version": KAP_VERSION, "created": _now(), "updated": _now(),
            "sections": {}, "dirty": False, "session_count": 0, "total_llm_calls": 0,
        }

def _save_state():
    _kap_state["updated"] = _now()
    with open(STATE_FILE, "w") as f:
        json.dump(_kap_state, f, indent=2)

def _load_log():
    global _kap_log
    if os.path.exists(LOG_FILE):
        try:
            with open(LOG_FILE, "r") as f:
                _kap_log = json.load(f)
        except Exception:
            pass

def _save_log():
    with open(LOG_FILE, "w") as f:
        json.dump(_kap_log, f, indent=2)

def _truncate(text, max_len, label=""):
    if len(text) <= max_len:
        return text
    return text[:max_len//2] + f"\n... [{label}] truncated {len(text)-max_len} chars ...\n" + text[-max_len//2:]

def _extract_code(text):
    """Robustly extract code from LLM markdown response."""
    text = text.strip()
    python_blocks = re.findall(r"```python\s*\n(.*?)\n\s*```", text, re.DOTALL)
    if python_blocks:
        return max(python_blocks, key=len).strip()
    generic_blocks = re.findall(r"```\s*\n(.*?)\n\s*```", text, re.DOTALL)
    if generic_blocks:
        return max(generic_blocks, key=len).strip()
    lines = text.splitlines()
    if lines and lines[0].strip().startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    if lines:
        return "\n".join(lines).strip()
    return text.strip()

def _call_llm(section_id, code, stdout, stderr, exc, attempt):
    return None  # MANUAL DEBUGGING: disable LLM calls
    
    
    # Rate limiting: wait at least LLM_COOLDOWN_SEC between calls
    elapsed = time.time() - _last_llm_call
    if elapsed < LLM_COOLDOWN_SEC:
        wait = LLM_COOLDOWN_SEC - elapsed
        print(f"  ⏳ Rate limit cooldown: waiting {wait:.1f}s...")
        time.sleep(wait)
    
    try:
        from openai import OpenAI
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])
        from openai import OpenAI

    client = OpenAI(api_key=GEMINI_API_KEY, base_url=GEMINI_BASE_URL)
    system = "You are an expert Python debugger. Fix the provided code. Return ONLY the complete fixed function inside a single ```python block. No explanations."

    code_send = _truncate(code, MAX_CODE_CHARS, "code")
    err_log = ""
    if exc.strip(): err_log += f"EXCEPTION:\n{exc}\n"
    if stderr.strip(): err_log += f"STDERR:\n{stderr}\n"
    err_log = _truncate(err_log, MAX_ERR_CHARS, "error")

    user = f"Section `{section_id}` failed (attempt {attempt}).\n\n---CODE---\n```python\n{code_send}\n```\n\n---ERROR---\n{err_log}\n\nReturn ONLY the complete fixed function inside ```python ... ```. No other text."

    try:
        _last_llm_call = time.time()
        resp = client.chat.completions.create(
            model=GEMINI_MODEL, messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ], temperature=0.1, max_tokens=64000, timeout=120)
        raw = resp.choices[0].message.content
        fixed = _extract_code(raw)
        if not fixed or fixed == raw.strip():
            print(f"  ⚠️  Could not extract code from LLM response. Raw length: {len(raw)}")
            return None
        return fixed
    except Exception as e:
        err_str = str(e)
        if "rate limit" in err_str.lower() or "429" in err_str or "quota" in err_str.lower():
            print(f"⚠️  Gemini rate limit hit. Waiting 30s before retry...")
            time.sleep(30)
            # Retry once
            try:
                _last_llm_call = time.time()
                resp = client.chat.completions.create(
                    model=GEMINI_MODEL, messages=[
                        {"role": "system", "content": system},
                        {"role": "user", "content": user},
                    ], temperature=0.1, max_tokens=64000, timeout=120)
                raw = resp.choices[0].message.content
                return _extract_code(raw)
            except Exception as e2:
                print(f"⚠️  Retry also failed: {e2}")
                return None
        print(f"⚠️  LLM API error: {e}")
        return None

# ── PUBLIC API ──

def init():
    global _kap_runner
    _load_state()
    _load_log()
    _kap_state["session_count"] = _kap_state.get("session_count", 0) + 1
    _save_state()

    sess = {
        "session_id": _ts(), "start_time": _now(), "end_time": None,
        "status": "RUNNING", "sections": [],
    }
    _kap_log["sessions"].append(sess)
    _save_log()
    _kap_runner = sess

    print(f"🚀 KAP {KAP_VERSION} initialized")
    print(f"   State: {STATE_FILE}")
    return sess

def run_section(section_id, func, resume=True, **kwargs):
    global _last_llm_call
    if _kap_runner is None:
        raise RuntimeError("Call init() first")

    sec = _kap_state["sections"].setdefault(section_id, {
        "status": "PENDING", "attempts": 0, "last_error": "", "last_run": None,
    })

    if resume and sec["status"] == "SUCCESS":
        print(f"[{section_id}] ⏩ SKIPPED (already passed)")
        return

    print(f"\n{'='*60}")
    print(f"[{section_id}] RUNNING: {func.__name__}")
    print(f"{'='*60}")

    for attempt in range(1, MAX_RETRIES + 1):
        sec["attempts"] += 1
        out = io.StringIO()
        err = io.StringIO()
        try:
            with redirect_stdout(out), redirect_stderr(err):
                result = func(**kwargs) if kwargs else func()
            ok = True
        except Exception:
            ok = False

        stdout = out.getvalue()
        stderr = err.getvalue()
        exc = traceback.format_exc() if not ok else ""

        if ok and not stderr.strip():
            print(f"[{section_id}] ✅ SUCCESS (attempt {attempt})")
            sec["status"] = "SUCCESS"
            sec["last_error"] = ""
            sec["last_run"] = _now()
            _save_state()
            entry = {"timestamp": _now(), "attempt": attempt, "success": True, "stdout": stdout[:2000], "stderr": stderr[:2000], "exception": "", "fix_applied": False}
            _kap_runner["sections"].append(entry)
            _save_log()
            return result

        print(f"[{section_id}] ❌ FAILED (attempt {attempt})")
        if exc:
            print(f"\n{'='*50}")
            print(f"[{section_id}] FULL TRACEBACK:")
            print(f"{'='*50}")
            for line in exc.strip().split('\n'):
                print(f"  {line}")
            print(f"{'='*50}")

        import inspect
        try:
            source = inspect.getsource(func)
        except Exception:
            source = f"# Source unavailable for {func.__name__}"

        print(f"[{section_id}] 🤖 Calling LLM for fix...")
        fixed = _call_llm(section_id, source, stdout, stderr, exc, attempt)
        _kap_state["total_llm_calls"] = _kap_state.get("total_llm_calls", 0) + 1
        _save_state()

        if not fixed:
            print(f"[{section_id}] ⚠️  No fix received. Stopping.")
            sec["status"] = "FAILED"
            sec["last_error"] = exc or stderr
            sec["last_run"] = _now()
            _save_state()
            entry = {"timestamp": _now(), "attempt": attempt, "success": False, "error": "LLM_NO_RESPONSE", "stdout": stdout[:2000], "stderr": stderr[:2000], "exception": exc[:2000], "fix_applied": False}
            _kap_runner["sections"].append(entry)
            _save_log()
            break

        if fixed.strip() == source.strip():
            print(f"[{section_id}] ⚠️  LLM returned unchanged code. Stopping.")
            sec["status"] = "FAILED"
            sec["last_error"] = exc or stderr
            sec["last_run"] = _now()
            _save_state()
            entry = {"timestamp": _now(), "attempt": attempt, "success": False, "error": "NO_CHANGE", "stdout": stdout[:2000], "stderr": stderr[:2000], "exception": exc[:2000], "fix_applied": False}
            _kap_runner["sections"].append(entry)
            _save_log()
            break

        # Save fix
        os.makedirs("/kaggle/working/kap_fixes", exist_ok=True)
        fix_path = f"/kaggle/working/kap_fixes/{section_id}_attempt{attempt}_{_ts()}.py"
        with open(fix_path, "w") as f:
            f.write(fixed)
        print(f"[{section_id}] 💾 Fix saved: {fix_path}")

        # Apply fix
        module = sys.modules.get(func.__module__)
        if module is None:
            module = sys.modules["__main__"]
        exec(fixed, module.__dict__)
        print(f"[{section_id}] 🔧 Fix applied, retrying...")
        _kap_state["dirty"] = True
        _save_state()

        entry = {"timestamp": _now(), "attempt": attempt, "success": False, "stdout": stdout[:2000], "stderr": stderr[:2000], "exception": exc[:2000], "fix_applied": True, "fix_path": fix_path}
        _kap_runner["sections"].append(entry)
        _save_log()
        time.sleep(2)
    else:
        print(f"[{section_id}] 🔥 MAX_RETRIES ({MAX_RETRIES}) reached")
        sec["status"] = "MAX_RETRIES"
        sec["last_error"] = exc or stderr
        sec["last_run"] = _now()
        _save_state()
        entry = {"timestamp": _now(), "attempt": MAX_RETRIES, "success": False, "error": "MAX_RETRIES", "stdout": stdout[:2000], "stderr": stderr[:2000], "exception": exc[:2000], "fix_applied": False}
        _kap_runner["sections"].append(entry)
        _save_log()

def final_report():
    if _kap_runner is None:
        raise RuntimeError("Call init() first")

    _kap_runner["end_time"] = _now()
    dirty = _kap_state.get("dirty", False)

    print("\n" + "=" * 60)
    print("KAP FINAL REPORT")
    print("=" * 60)

    ordered = ["S0", "S1", "S2", "S3", "S4"]
    for sid in ordered:
        sec = _kap_state["sections"].get(sid, {"status": "PENDING", "attempts": 0})
        status = sec["status"]
        emoji = {"SUCCESS": "✅", "FAILED": "❌", "MAX_RETRIES": "🔥", "PENDING": "⏸️", "RUNNING": "⏳"}.get(status, "❓")
        print(f"  {sid}: {emoji} {status} (attempts: {sec['attempts']})")

    print(f"\n  Dirty flag: {'True → RE-COMMIT' if dirty else 'False → stable'}")
    print(f"  LLM calls: {_kap_state.get('total_llm_calls', 0)}")
    print("=" * 60)

    if dirty:
        print("\n⚠️  FIXES APPLIED. Re-commit this notebook to test them.")

    all_pass = all(_kap_state["sections"].get(sid, {}).get("status") == "SUCCESS" for sid in ordered)
    status = "SUCCESS" if all_pass else "PARTIAL" if any(_kap_state["sections"].get(sid, {}).get("status") == "SUCCESS" for sid in ordered) else "FAILED"
    _kap_runner["status"] = status
    _save_log()
    return status

def is_dirty():
    return _kap_state.get("dirty", False)

# Auto-init on import
init()

In [ ]:
#================================================================================
def section_S0():  # Environment Setup (cells 4-9)
#================================================================================

    # ## SECTION 0: Environment Setup
    #
    # Two install paths, deliberately kept apart — this is the actual fix for the Colab crash:
    #
    # - **pip**, straight into the kernel's own Python — OpenMM, ParmEd, MDAnalysis, BioPython, RDKit, Vina, ProLIF, etc. Kaggle has one Python env, so there's nothing for these to mismatch against.
    # - **Miniconda in userspace**, used only to get the `packmol-memgen`/`tleap` CLI binaries from the `ambermd` conda channel (AmberTools has no pip wheel). We never `import` from this env in the kernel — only call its binaries as subprocesses — so there's no mismatch possible here either.
    #
    # No restart needed for either step.

    # Step 0a: pip-install everything that has a wheel, directly into the kernel's own Python.
    # This is the fix for the Colab python3.11/3.12 mismatch — no conda, no separate env, no restart.
    import sys, subprocess, time
    
    t0 = time.time()
    print('Installing for:', sys.executable)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'openmm', 'openmmforcefields', 'pdbfixer', 'parmed',
         'mdanalysis', 'biopython', 'rdkit', 'vina', 'prolif',
         'matplotlib', 'seaborn', 'pandas', 'numpy', 'scipy'],
        check=True,
    )
    print(f'pip install done in {time.time()-t0:.0f}s')

    # Step 0b: Miniconda in userspace, AmberTools only, for the packmol-memgen / tleap CLI.
    # We deliberately do NOT activate this env or import from it in this kernel —
    # only its binaries are called by full path via subprocess (SECTION 5 below).
    #
    # IMPORTANT: the conda prefix and installer MUST live OUTSIDE /kaggle/working.
    # /kaggle/working is what Kaggle commits as the notebook's Output on Save & Run All —
    # a full Miniconda + AmberTools env is thousands of files / 1-3+ GB, which blew the
    # output size/count limit on Version 4 (cells ran fine, but the post-execution commit
    # silently failed - logs showed nbconvert succeeding with no Python traceback).
    import os, subprocess, time
    
    CONDA_PREFIX = '/opt/miniconda'
    AMBER_ENV = f'{CONDA_PREFIX}/envs/ambertools'
    
    if not os.path.exists(f'{AMBER_ENV}/bin/packmol-memgen'):
        t0 = time.time()
        subprocess.run(['wget', '-q',
                         'https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh',
                         '-O', '/tmp/miniconda.sh'], check=True)
        subprocess.run(['bash', '/tmp/miniconda.sh', '-b', '-f', '-p', CONDA_PREFIX], check=True)
        os.remove('/tmp/miniconda.sh')
        conda_bin = f'{CONDA_PREFIX}/bin/conda'
        subprocess.run([conda_bin, 'tos', 'accept', '--override-channels', '--channel', 'https://repo.anaconda.com/pkgs/main'])
        subprocess.run([conda_bin, 'tos', 'accept', '--override-channels', '--channel', 'https://repo.anaconda.com/pkgs/r'])
        subprocess.run([conda_bin, 'create', '-y', '-q', '-n', 'ambertools',
                         '-c', 'conda-forge', '-c', 'ambermd', 'ambertools'], check=True)
        subprocess.run([conda_bin, 'clean', '-a', '-y'], check=True)  # drop package cache, keep output lean
        print(f'AmberTools env built in {time.time()-t0:.0f}s')
    else:
        print('AmberTools env already present, skipping')
    
    PACKMOL_MEMGEN = f'{AMBER_ENV}/bin/packmol-memgen'
    assert os.path.exists(PACKMOL_MEMGEN), 'packmol-memgen not found — check the conda build log above'
    print('packmol-memgen at:', PACKMOL_MEMGEN)

    import json
    with open('gpr151_pipeline/env_config.json', 'w') as f:
        json.dump({
            'AMBER_ENV': AMBER_ENV,
            'PACKMOL_MEMGEN': PACKMOL_MEMGEN,
            'CONDA_ENV': CONDA_ENV
        }, f)
    print(f'💾 Saved env config: AMBER_ENV={AMBER_ENV}, PACKMOL={PACKMOL_MEMGEN}')

    # Step 0c: Verify GPU and imports (should be instant — no conda/kernel mismatch to hang on)
    import subprocess
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                            capture_output=True, text=True)
    print('GPU:', result.stdout.strip())
    
    import openmm
    from openmm import Platform
    for i in range(Platform.getNumPlatforms()):
        print(f'  OpenMM platform: {Platform.getPlatform(i).getName()}')
    
    import os
    os.makedirs('/kaggle/working/gpr151_pipeline', exist_ok=True)
    print('\n✅ Environment ready')

    # ## SECTION 0.5: Resume from a previous Kaggle session (optional)
    #
    # Kaggle sessions cap at ~9h, so SECTION 8 (production MD) is checkpointed exactly like it was
    # checkpointed to Google Drive in the Colab version — same chunk size, same file names — just
    # carried between sessions via a Kaggle Dataset instead of a Drive folder:
    #
    # 1. After a commit finishes, open its **Output** tab -> **New Dataset** to publish
    # `gpr151_pipeline/` (e.g. as `gpr151-checkpoint`).
    # 2. In the next session, **Add Data -> Your Datasets -> gpr151-checkpoint** (mounts read-only
    # at `/kaggle/input/gpr151-checkpoint/`).
    # 3. Run the cell below to copy any checkpoint/trajectory files into the writable
    # `/kaggle/working/gpr151_pipeline/` — SECTION 8 auto-detects `steps_completed.txt` and resumes.
    #
    # Skip this cell on the first session.

    import glob, shutil, os
    
    RESUME_INPUT = '/kaggle/input/gpr151-checkpoint'  # rename to match your dataset slug
    if os.path.isdir(RESUME_INPUT):
        copied = 0
        for f in glob.glob(f'{RESUME_INPUT}/**/*', recursive=True):
            if os.path.isfile(f):
                dest = f.replace(RESUME_INPUT, '/kaggle/working/gpr151_pipeline')
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                shutil.copy2(f, dest)
                copied += 1
        print(f'✅ Restored {copied} files from previous session')
    else:
        print('No resume dataset attached — starting fresh')


In [ ]:

#================================================================================
def section_S1():  # Fetch & Prep Structure (cells 10-15)
#================================================================================

    # ## SECTION 1: Fetch GPR151 AlphaFold Structure
    # We use the wild-type human GPR151 AlphaFold model (UniProt Q8TDV0) rather than the NELiS cryo-EM structure, which contains 4 thermostabilizing mutations and a bound nanobody. The WT AlphaFold model gives us the native sequence for drug discovery without confounding mutations.

    import requests, os
    os.chdir('/kaggle/working')
    import os
    import json
    
    os.makedirs('gpr151_pipeline', exist_ok=True)
    
    # Download AlphaFold v4 model for human GPR151 (UniProt Q8TDV0)
    AF_URL = 'https://alphafold.ebi.ac.uk/files/AF-Q8TDV0-F1-model_v6.pdb'
    print(f'Fetching AlphaFold structure: {AF_URL}')
    
    r = requests.get(AF_URL, timeout=60)
    r.raise_for_status()
    
    with open('gpr151_pipeline/gpr151_alphafold.pdb', 'w') as f:
        f.write(r.text)
    
    # Count residues
    lines = [l for l in r.text.split('\n') if l.startswith('ATOM') and l[12:16].strip() == 'CA']
    residue_count = len(lines)
    print(f'✅ Downloaded GPR151 AlphaFold model: {residue_count} residues')
    
    # Also fetch AlphaFold PAE (confidence) data to identify low-confidence N-terminal region
    PAE_URL = 'https://alphafold.ebi.ac.uk/files/AF-Q8TDV0-F1-predicted_aligned_error_v6.json'
    r_pae = requests.get(PAE_URL, timeout=60)
    if r_pae.ok:
        pae_data = r_pae.json()
        print(f'✅ Downloaded PAE confidence data')
        with open('gpr151_pipeline/gpr151_pae.json', 'w') as f:
            json.dump(pae_data, f)
    else:
        print('PAE data not available, will use pLDDT from PDB B-factors')

    # Also fetch UniProt sequence and metadata for GPR151
    UNIPROT_URL = 'https://rest.uniprot.org/uniprotkb/Q8TDV0.json'
    r_up = requests.get(UNIPROT_URL, timeout=30)
    uniprot_data = r_up.json()
    
    sequence = uniprot_data['sequence']['value']
    seq_length = uniprot_data['sequence']['length']
    print(f'GPR151 sequence length: {seq_length} residues')
    print(f'First 60 aa (N-terminus region): {sequence[:60]}')
    print(f'Gene name: {uniprot_data["genes"][0]["geneName"]["value"]}')
    
    # Extract annotated transmembrane regions from UniProt
    tm_regions = []
    for feature in uniprot_data.get('features', []):
        if feature.get('type') == 'Transmembrane':
            start = feature['location']['start']['value']
            end = feature['location']['end']['value']
            desc = feature.get('description', '')
            tm_regions.append((start, end, desc))
            print(f'  TM region: residues {start}–{end} ({desc})')
    
    if tm_regions:
        TM1_START = tm_regions[0][0]  # First TM helix start
        print(f'\n📌 TM1 starts at residue {TM1_START}')
        print(f'   N-terminal autoinhibitory region: residues 1–{TM1_START - 1}')
    else:
        print('UniProt TM annotation not found — will use GPCRDB in next cell')

    # ## SECTION 2: GPCRDB TM Helix Annotation
    # Query the GPCRDB API for GPR151 generic numbering to precisely define TM1 start and validate the truncation boundary.

    import requests
    import pandas as pd
    import matplotlib.pyplot as plt
    import numpy as np
    
    # Query GPCRDB for GPR151 residue annotations
    GPCRDB_URL = 'https://gpcrdb.org/services/residues/extended/gpr151_human/'
    print(f'Querying GPCRDB: {GPCRDB_URL}')
    
    try:
        r_gpcr = requests.get(GPCRDB_URL, timeout=30)
        residues = r_gpcr.json()
    
        # Build dataframe of residues with segment annotation
        df_res = pd.DataFrame(residues)
        print(f'Retrieved {len(df_res)} residue annotations')
        print(f'Segments found: {df_res["protein_segment"].unique().tolist()}')
    
        # Find TM1 boundaries
        tm1 = df_res[df_res['protein_segment'] == 'TM1']
        if len(tm1) > 0:
            GPCRDB_TM1_START = int(tm1['sequence_number'].min())
            GPCRDB_TM1_END = int(tm1['sequence_number'].max())
            print(f'\n📌 GPCRDB TM1: residues {GPCRDB_TM1_START}–{GPCRDB_TM1_END}')
    
            # Show all TM helices
            for tm in ['TM1','TM2','TM3','TM4','TM5','TM6','TM7']:
                seg = df_res[df_res['protein_segment'] == tm]
                if len(seg) > 0:
                    print(f'  {tm}: {int(seg["sequence_number"].min())}–{int(seg["sequence_number"].max())}')
    
            # N-terminus (before TM1)
            nterm = df_res[df_res['protein_segment'] == 'N-term']
            print(f'\n  N-terminus annotated: {len(nterm)} residues (1–{int(nterm["sequence_number"].max()) if len(nterm)>0 else GPCRDB_TM1_START-1})')
    
            # Use GPCRDB annotation if available, else UniProt
            TM1_START = GPCRDB_TM1_START
    
        # Save for reference
        df_res.to_csv('gpr151_pipeline/gpcrdb_residues.csv', index=False)

        # Save TM1_START for downstream sections (S2, S3, S4)
    
    
    except Exception as e:
        print(f'GPCRDB query failed: {e}')
        print('Falling back to UniProt TM annotation')
        
        # TM1_START already set from UniProt in Section 1
    import json
    with open('gpr151_pipeline/tm1_start.json', 'w') as f:
        json.dump({'TM1_START': TM1_START}, f)
    print(f'💾 Saved TM1_START={TM1_START} for downstream sections')
    
    
    # --- Manual override: cut at residue 32, not the sequence-topology-based TM1_START ---
    # GPCRDB returned 0 residues for gpr151_human (query failed), and UniProt's
    # Transmembrane(42-62) annotation is a sequence-based topology prediction
    # (Phobius/TMHMM-style), not a measurement of the actual AlphaFold structure.
    #
    # A real per-residue check of the AlphaFold model (pLDDT + secondary structure,
    # via biotite.structure.annotate_sse) shows:
    #   - res 1-10:  pLDDT 35-43 (very low) -> genuinely disordered, likely the real
    #                autoinhibitory arm that occludes the orthosteric pocket
    #   - res 11-20: pLDDT 34-56, called "helix" by the SSE algorithm, but at this
    #                confidence that call is not trustworthy (spurious)
    #   - res 21-31: pLDDT 60-78, noisy transitional coil
    #   - res 32-37: pLDDT 71-91 (confident -> very high), well-ordered coil/loop
    #                feeding into TM1 -- legitimate structure, not autoinhibitory junk
    #   - res 38+:   pLDDT 92+, the TM1 helix proper (UniProt's "42" call is ~4
    #                residues into the actual helix, which would have truncated
    #                into real secondary structure)
    #
    # Cutting at 32 removes only the genuinely disordered/low-confidence segment
    # (including the spurious low-confidence "helix" at 11-20) while preserving
    # the ordered loop (32-37) and the full native TM1 helix (38+) intact.
    TM1_START = 32
    
    print(f'\n✅ Final truncation point: residue {TM1_START} (first residue of TM1 helix)')
    print(f'   Removing N-terminal residues 1–{TM1_START - 1}')

    # Plot pLDDT confidence scores along the sequence
    # Low pLDDT in N-terminus confirms disordered/flexible autoinhibitory region
    from Bio import PDB
    
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('gpr151', 'gpr151_pipeline/gpr151_alphafold.pdb')
    
    residue_ids = []
    plddt_scores = []
    for model in structure:
        for chain in model:
            for residue in chain:
                res_id = residue.get_id()[1]
                # In AlphaFold PDBs, B-factor column = pLDDT
                ca_atoms = [a for a in residue if a.get_name() == 'CA']
                if ca_atoms:
                    residue_ids.append(res_id)
                    plddt_scores.append(ca_atoms[0].get_bfactor())
    
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(residue_ids, plddt_scores, color='steelblue', linewidth=0.8)
    ax.axvline(x=TM1_START, color='red', linestyle='--', linewidth=2, label=f'Truncation site (res {TM1_START})')
    ax.axhspan(0, 70, alpha=0.25, color='r', label='Low/OK confidence (<70)')
    ax.axhspan(70, 90, alpha=0.1, color='y', label='Confident (70–90)')
    ax.axhspan(90, 100, alpha=0.1, color='g', label='Very confident (>90)')
    ax.set_xlabel('Residue number')
    ax.set_ylabel('pLDDT confidence')
    ax.set_title('GPR151 AlphaFold pLDDT — low N-terminal confidence confirms flexible autoinhibitory region')
    ax.legend(loc='lower right', fontsize=8)
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.savefig('gpr151_pipeline/plddt_plot.png', dpi=150)
    plt.show()
    print(f'N-terminal pLDDT (res 1–{TM1_START-1}): {np.mean(plddt_scores[:TM1_START-1]):.1f}')
    print(f'TM domain pLDDT (res {TM1_START}+): {np.mean(plddt_scores[TM1_START-1:]):.1f}')


In [ ]:
def section_S2():

    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pdbfixer"])

    # SECTION 3: N-Terminal Truncation
    import json
    with open('gpr151_pipeline/env_config.json') as f:
        env_cfg = json.load(f)
    AMBER_ENV = env_cfg['AMBER_ENV']
    PACKMOL_MEMGEN = env_cfg['PACKMOL_MEMGEN']
    print(f'📂 Loaded env: AMBER_ENV={AMBER_ENV}, PACKMOL={PACKMOL_MEMGEN}')
    with open('gpr151_pipeline/tm1_start.json') as f:
        TM1_START = json.load(f)['TM1_START']
    print(f'📂 Loaded TM1_START={TM1_START} from S1')

    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "biopython"])

    from Bio.PDB import PDBParser, PDBIO, Select
    import os
    
    class NTermTruncationSelect(Select):
        def __init__(self, tm1_start):
            self.tm1_start = tm1_start
        def accept_residue(self, residue):
            res_id = residue.get_id()[1]
            return res_id is not None and res_id >= self.tm1_start
    
    parser = PDBParser(QUIET=True)
    pdb_path = 'gpr151_pipeline/gpr151_alphafold.pdb'
    
    if not os.path.exists(pdb_path):
        raise FileNotFoundError(f"Missing: {pdb_path}. Run S1 first.")
    
    structure = parser.get_structure('gpr151', pdb_path)
    all_res = list(structure.get_residues())
    kept = [r for r in all_res if r.get_id()[1] >= TM1_START]
    
    print(f'Original: {len(all_res)} residues')
    print(f'GPR151-ΔN: {len(kept)} residues (removed 1-{TM1_START-1})')
    
    io = PDBIO()
    io.set_structure(structure)
    io.save('gpr151_pipeline/gpr151_delta_n.pdb', NTermTruncationSelect(TM1_START))
    print(f'✅ Saved truncated structure')

    # SECTION 4: PDBFixer
    from pdbfixer import PDBFixer
    from openmm.app import PDBFile
    
    fixer = PDBFixer(filename='gpr151_pipeline/gpr151_delta_n.pdb')
    fixer.findMissingResidues()
    # Remove gaps before TM1_START
    for k in list(fixer.missingResidues.keys()):
        if k[1] < TM1_START:
            del fixer.missingResidues[k]
    
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.removeHeterogens(keepWater=False)
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.4)
    
    with open('gpr151_pipeline/gpr151_prepared.pdb', 'w') as f:
        PDBFile.writeFile(fixer.topology, fixer.positions, f)
    
    print(f'✅ Prepared: {fixer.topology.getNumResidues()} residues')

    # SECTION 5: Membrane System
    import subprocess, glob
    
    amber_env = os.environ.copy()
    amber_env["AMBERHOME"] = AMBER_ENV
    amber_env["PATH"] = f"{AMBER_ENV}/bin:" + amber_env.get("PATH", "")
    
    cmd = [
        PACKMOL_MEMGEN,
        "--pdb", "gpr151_prepared.pdb",
        "--lipids", "POPC:POPE:CHL1", "--ratio", "2:2:1",
        "--distxy_fix", "80", "--dist", "10", "--dist_wat", "17.5",
        "--nloop", "20", "--nloop_all", "100",
        "--notprotonate",
    ]
    
    print(f"Running packmol-memgen...")
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=900, env=amber_env)
    
    print(f"Exit code: {result.returncode}")
    print(result.stdout[-2000:])
    if result.stderr:
        print("STDERR:", result.stderr[-1000:])
    
    if result.returncode != 0:
        raise RuntimeError(f'packmol-memgen failed: {result.returncode}')
    
    # Find outputs
    files = os.listdir('gpr151_pipeline')
    print(f"Files created: {files}")
    
    pdb_files = [f for f in files if f.endswith('.pdb')]
    if pdb_files:
        global MEMBRANE_PDB
        MEMBRANE_PDB = pdb_files[0]
        print(f'✅ Membrane system: {MEMBRANE_PDB}')
    else:
        raise FileNotFoundError('No membrane PDB created')


In [ ]:
#================================================================================
def section_S3():  # Production MD (cells 23-33)
#================================================================================

    # ## SECTION 6: OpenMM Simulation Setup
    #
    # Load the membrane system via **ParmEd** (reads the CHARMM-style whole-residue names
    # `POPC`/`POPE`/`CHL1`/`TIP3` that packmol-memgen writes -- `tleap` truncates these to 3 characters
    # and fails with `Unknown residue: POP`), build the OpenMM `System` with the native **CHARMM36m**
    # force field (already defines these residues, no AMBER lipid parameters needed), configure the
    # membrane-specific NPT barostat, and set up the Langevin integrator with hydrogen mass
    # repartitioning for a 4 fs timestep.
    #
    # This is the fix verified against the actual bilayer output during the Colab debugging session,
    # ported unchanged -- the bug it works around (tleap + CHARMM-style residue names) is independent
    # of which notebook platform runs it.

    import subprocess, os
    from openmm.app import ForceField, PME, HBonds, Simulation, AmberPrmtopFile, AmberInpcrdFile, PDBFile
    from openmm import Platform
    from openmm.unit import nanometer, nanometers, amu, bar, kelvin, picosecond, picoseconds, kilojoules_per_mole
    from openmm import MonteCarloMembraneBarostat, LangevinMiddleIntegrator

    BILAYER_PDB = f'gpr151_pipeline/{MEMBRANE_PDB}'
    print(f'Building system via tleap (AMBER ff19SB + Lipid21): {BILAYER_PDB}')

    # Fix: packmol-memgen's Lipid21 output (CHL1 etc.) doesn't match OpenMM's ported
    # amber19/lipid21.xml CHL1 template -- confirmed via V18 traceback ("User-supplied
    # template CHL1 does not match the residue"), even with an explicit residueTemplates=
    # override. Build the prmtop/inpcrd directly with tleap (AmberTools' own native
    # Lipid21 parameters, which DO match this PDB) instead of going through OpenMM's
    # ForceField template matcher at all.
    with open(BILAYER_PDB) as f:
        _lines = f.readlines()
    _cryst = [l for l in _lines if l.startswith('CRYST1')]
    if _cryst:
        box_a, box_b, box_c = (float(x) for x in _cryst[0].split()[1:4])
    else:
        # packmol-memgen didn't write a CRYST1 record for this --distxy_fix run --
        # derive the box straight from the packed atom coordinates instead.
        print('  No CRYST1 record found -- computing box from atom coordinates')
        _xs, _ys, _zs = [], [], []
        for l in _lines:
            if l.startswith('ATOM') or l.startswith('HETATM'):
                _xs.append(float(l[30:38]))
                _ys.append(float(l[38:46]))
                _zs.append(float(l[46:54]))
        box_a = max(_xs) - min(_xs)
        box_b = max(_ys) - min(_ys)
        box_c = max(_zs) - min(_zs)
    print(f'  Box: {box_a:.2f} x {box_b:.2f} x {box_c:.2f} A')

    leap_script = f"""source leaprc.protein.ff19SB
source leaprc.water.tip3p
source leaprc.lipid21
mol = loadpdb {BILAYER_PDB}
set mol box {{ {box_a} {box_b} {box_c} }}
saveamberparm mol gpr151_pipeline/system.prmtop gpr151_pipeline/system.inpcrd
quit
"""
    with open('gpr151_pipeline/build.leap', 'w') as f:
        f.write(leap_script)

    env = dict(os.environ)
    env['AMBERHOME'] = '/opt/miniconda/envs/ambertools'
    result = subprocess.run(
        ['/opt/miniconda/envs/ambertools/bin/tleap', '-f', 'gpr151_pipeline/build.leap'],
        capture_output=True, text=True, env=env,
    )
    print(result.stdout[-2000:])
    if result.returncode != 0 or not os.path.exists('gpr151_pipeline/system.prmtop'):
        raise RuntimeError(f'tleap failed:\n{result.stdout}\n{result.stderr}')

    prmtop = AmberPrmtopFile('gpr151_pipeline/system.prmtop')
    inpcrd = AmberInpcrdFile('gpr151_pipeline/system.inpcrd')
    topology = prmtop.topology
    positions = inpcrd.positions
    box_vectors = inpcrd.boxVectors

    print('Creating OpenMM system...')
    system = prmtop.createSystem(
        nonbondedMethod=PME, nonbondedCutoff=1.2 * nanometer,
        constraints=HBonds, hydrogenMass=1.5 * amu, rigidWater=True, removeCMMotion=True,
    )
    print(f'  System: {system.getNumParticles():,} particles')

    barostat = MonteCarloMembraneBarostat(
        1.0 * bar, 0.0 * bar * nanometer, 310.0 * kelvin,
        MonteCarloMembraneBarostat.XYIsotropic, MonteCarloMembraneBarostat.ZFree, 25,
    )
    system.addForce(barostat)

    integrator = LangevinMiddleIntegrator(310.0 * kelvin, 1.0 / picosecond, 0.004 * picosecond)
    integrator.setRandomNumberSeed(42)

    try:
        platform = Platform.getPlatformByName('CUDA')
        properties = {'CudaPrecision': 'mixed'}
        print('Using CUDA GPU platform')
    except Exception:
        try:
            platform = Platform.getPlatformByName('OpenCL')
            properties = {}
            print('Using OpenCL platform')
        except Exception:
            platform = Platform.getPlatformByName('CPU')
            properties = {}
            print('WARNING: CPU platform -- check accelerator setting in Kaggle Settings sidebar')

    simulation = Simulation(topology, system, integrator, platform, properties)
    simulation.context.setPositions(positions)
    if box_vectors is not None:
        simulation.context.setPeriodicBoxVectors(*box_vectors)

    print(f'\n✅ OpenMM simulation created: {system.getNumParticles():,} particles')

    # ## SECTION 7: Energy Minimization + Equilibration
    # Minimization removes steric clashes from the membrane builder. Equilibration runs in stages with progressively released restraints on protein heavy atoms.

    import time

    # --- Energy Minimization ---
    print('Step 1: Energy minimization...')
    t0 = time.time()

    initial_energy = simulation.context.getState(getEnergy=True).getPotentialEnergy()
    print(f'  Initial energy: {initial_energy.value_in_unit(kilojoules_per_mole):.1f} kJ/mol')

    simulation.minimizeEnergy(tolerance=10.0*kilojoules_per_mole, maxIterations=5000)

    final_energy = simulation.context.getState(getEnergy=True).getPotentialEnergy()
    print(f'  Final energy: {final_energy.value_in_unit(kilojoules_per_mole):.1f} kJ/mol')
    print(f'  Time: {time.time()-t0:.1f}s')

    # Save minimized structure
    state = simulation.context.getState(getPositions=True)
    with open('gpr151_pipeline/gpr151_minimized.pdb', 'w') as f:
        PDBFile.writeFile(simulation.topology, state.getPositions(), f)
    print('  Saved: gpr151_minimized.pdb')

    # --- Equilibration Phase 1: NVT with protein restraints (1 ns) ---
    # Apply positional restraints to protein heavy atoms
    from openmm import CustomExternalForce

    def add_protein_restraints(system, topology, positions, k=1000.0):
        """Add flat-bottom positional restraints to protein heavy atoms."""
        restraint = CustomExternalForce('k*((x-x0)^2+(y-y0)^2+(z-z0)^2)')
        restraint.addGlobalParameter('k', k * kilojoules_per_mole/nanometer**2)
        restraint.addPerParticleParameter('x0')
        restraint.addPerParticleParameter('y0')
        restraint.addPerParticleParameter('z0')

        protein_residues = {'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','HIE','HID',
                            'ILE','LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'}
        n_restrained = 0
        for atom in topology.atoms():
            if atom.residue.name in protein_residues and atom.element.symbol != 'H':
                pos = positions[atom.index].value_in_unit(nanometers)
                restraint.addParticle(atom.index, pos)
                n_restrained += 1
        system.addForce(restraint)
        return n_restrained

    state = simulation.context.getState(getPositions=True)
    n_restrained = add_protein_restraints(system, topology, state.getPositions())
    simulation.context.reinitialize(preserveState=True)
    print(f'Applied restraints to {n_restrained} protein heavy atoms')

    # Set initial velocities
    simulation.context.setVelocitiesToTemperature(310*kelvin, 42)

    # Reporters
    from openmm.app import StateDataReporter, DCDReporter

    simulation.reporters.append(StateDataReporter(
        'gpr151_pipeline/equilibration.log', 2500,  # every 10 ps
        step=True, time=True, potentialEnergy=True, kineticEnergy=True,
        totalEnergy=True, temperature=True, volume=True, progress=True,
        remainingTime=True, speed=True, totalSteps=250000))

    print('\nStep 2: NVT equilibration with protein restraints (1 ns)...')
    t0 = time.time()
    simulation.step(250000)  # 1 ns at 4 fs/step
    print(f'  Done in {time.time()-t0:.0f}s')

    state = simulation.context.getState(getPositions=True)
    with open('gpr151_pipeline/gpr151_nvt.pdb', 'w') as f:
        PDBFile.writeFile(simulation.topology, state.getPositions(), f)
    print('  Saved: gpr151_nvt.pdb')

    # --- Equilibration Phase 2: NPT with reduced restraints (2 ns) ---
    # Remove restraints from the system (replace with weaker ones)
    # Re-create system without restraints but with barostat
    # FIX: use prmtop.createSystem instead of undefined forcefield.createSystem
    system2 = prmtop.createSystem(
        nonbondedMethod=PME,
        nonbondedCutoff=1.2*nanometers,
        constraints=HBonds,
        hydrogenMass=1.5*amu
    )
    system2.addForce(MonteCarloMembraneBarostat(
        1.0*bar, 0.0*bar*nanometer, 310.0*kelvin,
        MonteCarloMembraneBarostat.XYIsotropic, MonteCarloMembraneBarostat.ZFree, 25))

    # Add weak (50 kJ/mol/nm²) restraints for NPT equilibration
    state = simulation.context.getState(getPositions=True, getVelocities=True)
    n_restrained = add_protein_restraints(system2, topology, state.getPositions(), k=50.0)

    integrator2 = LangevinMiddleIntegrator(310.0*kelvin, 1.0/picosecond, 0.004*picoseconds)
    integrator2.setRandomNumberSeed(43)

    simulation2 = Simulation(topology, system2, integrator2, platform, properties)
    simulation2.context.setPositions(state.getPositions())
    simulation2.context.setVelocities(state.getVelocities())
    if state.getPeriodicBoxVectors() is not None:
        simulation2.context.setPeriodicBoxVectors(*state.getPeriodicBoxVectors())

    simulation2.reporters.append(StateDataReporter(
        'gpr151_pipeline/npt_equil.log', 2500,
        step=True, time=True, potentialEnergy=True, temperature=True,
        volume=True, density=True, progress=True, remainingTime=True,
        speed=True, totalSteps=500000))

    print('Step 3: NPT equilibration with weak restraints (2 ns)...')
    t0 = time.time()
    simulation2.step(500000)  # 2 ns
    print(f'  Done in {time.time()-t0:.0f}s')

    state2 = simulation2.context.getState(getPositions=True, getVelocities=True)
    with open('gpr151_pipeline/gpr151_npt.pdb', 'w') as f:
        PDBFile.writeFile(simulation2.topology, state2.getPositions(), f)
    print('  Saved: gpr151_npt.pdb')

    # ## SECTION 8: Production MD -- Checkpoint-Resume (Kaggle Datasets)
    #
    # Same design as the Colab version's Google-Drive checkpointing -- 5 ns chunks (CHUNK_STEPS =
    # 1,250,000), same file names (checkpoint.chk, chunk_NNN.dcd, production.log,
    # steps_completed.txt) -- just persisted via a Kaggle Dataset across sessions instead of a Drive
    # folder (see SECTION 0.5 for the resume procedure). Within a single ~9h session this just
    # accumulates chunks in `/kaggle/working/gpr151_pipeline/` directly; only carrying state *between*
    # sessions needs the publish-as-dataset step.

    import os, time, glob, shutil

    WORK_DIR = 'gpr151_pipeline'  # was DRIVE_DIR in the Colab version -- Kaggle persists this via
                                   # Datasets across sessions instead of via a Drive mount
    os.makedirs(WORK_DIR, exist_ok=True)

    # Production MD settings -- identical to the Colab version
    PROD_NS      = 50           # total nanoseconds to simulate
    STEPS_PER_NS = 250_000      # 4 fs timestep -> 250k steps/ns
    TOTAL_STEPS  = PROD_NS * STEPS_PER_NS
    SAVE_EVERY   = 25_000       # trajectory frame every 100 ps
    CHUNK_STEPS  = 1_250_000    # 5 ns per chunk (safe within a ~9h Kaggle session)

    progress_file   = f'{WORK_DIR}/steps_completed.txt'
    checkpoint_file = f'{WORK_DIR}/checkpoint.chk'

    steps_done = 0
    if os.path.exists(progress_file):
        with open(progress_file) as f:
            steps_done = int(f.read().strip())
        print(f'Resuming: {steps_done/STEPS_PER_NS:.1f} ns done '
              f'({steps_done:,}/{TOTAL_STEPS:,} steps)')
    else:
        print('Fresh start -- no prior checkpoint found')

    if steps_done >= TOTAL_STEPS:
        print(f'\n✅ Production MD already complete ({PROD_NS} ns). Skip to Section 9.')
    else:
        ns_left = (TOTAL_STEPS - steps_done) / STEPS_PER_NS
        print(f'Remaining: {ns_left:.1f} ns')

    from openmm.app import CheckpointReporter

    if steps_done < TOTAL_STEPS:
        # FIX: use prmtop.createSystem instead of undefined forcefield.createSystem
        system_prod = prmtop.createSystem(
            topology,
            nonbondedMethod=PME,
            nonbondedCutoff=1.2*nanometers,
            constraints=HBonds,
            hydrogenMass=1.5*amu
        )
        system_prod.addForce(MonteCarloMembraneBarostat(
            1.0*bar, 0.0*bar*nanometer, 310.0*kelvin,
            MonteCarloMembraneBarostat.XYIsotropic,
            MonteCarloMembraneBarostat.ZFree, 25))

        integrator_prod = LangevinMiddleIntegrator(
            310.0*kelvin, 1.0/picosecond, 0.004*picoseconds)
        integrator_prod.setRandomNumberSeed(44)

        try:
            platform = Platform.getPlatformByName('CUDA')
            properties = {'CudaPrecision': 'mixed'}
            print('CUDA platform')
        except Exception:
            platform = Platform.getPlatformByName('OpenCL')
            properties = {}
            print('OpenCL platform')

        sim_prod = Simulation(topology, system_prod,
                              integrator_prod, platform, properties)

        if os.path.exists(checkpoint_file) and steps_done > 0:
            print('Loading checkpoint...')
            sim_prod.loadCheckpoint(checkpoint_file)
            print(f'Resumed at {steps_done/STEPS_PER_NS:.1f} ns')
        else:
            sim_prod.context.setPositions(state2.getPositions())
            sim_prod.context.setVelocities(state2.getVelocities())
            if state2.getPeriodicBoxVectors() is not None:
                sim_prod.context.setPeriodicBoxVectors(*state2.getPeriodicBoxVectors())
            print('Starting from equilibrated NPT state')

        print('✅ Simulation ready')

    import time

    if steps_done < TOTAL_STEPS:
        steps_left     = TOTAL_STEPS - steps_done
        steps_this_run = min(CHUNK_STEPS, steps_left)
        chunk_num      = steps_done // CHUNK_STEPS + 1
        dcd_local      = f'{WORK_DIR}/chunk_{chunk_num:03d}.dcd'
        log_local      = f'{WORK_DIR}/production.log'

        sim_prod.reporters.clear()
        sim_prod.reporters.append(DCDReporter(dcd_local, SAVE_EVERY))
        sim_prod.reporters.append(StateDataReporter(
            log_local, SAVE_EVERY,
            step=True, time=True, potentialEnergy=True, kineticEnergy=True,
            temperature=True, volume=True, density=True,
            progress=True, remainingTime=True, speed=True,
            totalSteps=TOTAL_STEPS, separator='\t',
            append=os.path.exists(log_local)))
        sim_prod.reporters.append(CheckpointReporter(checkpoint_file, STEPS_PER_NS * 2))

        ns_this = steps_this_run / STEPS_PER_NS
        print(f'Chunk {chunk_num}: {steps_done/STEPS_PER_NS:.1f} -> '
              f'{(steps_done+steps_this_run)/STEPS_PER_NS:.1f} ns')
        t0 = time.time()

        sim_prod.step(steps_this_run)

        elapsed = time.time() - t0
        rate    = ns_this / (elapsed / 86400)
        print(f'   Done in {elapsed/60:.1f} min ({rate:.0f} ns/day)')

        new_steps = steps_done + steps_this_run
        with open(progress_file, 'w') as fp:
            fp.write(str(new_steps))

        pct = new_steps / TOTAL_STEPS * 100
        print(f'\n✅ Progress: {new_steps/STEPS_PER_NS:.1f}/{PROD_NS} ns ({pct:.0f}%)')

        if new_steps >= TOTAL_STEPS:
            print('\nAll 50 ns complete! Proceed to the next cell to assemble trajectory.')
        else:
            remaining = (TOTAL_STEPS - new_steps) / STEPS_PER_NS
            print(f'\n{remaining:.1f} ns remaining.')
            print('   Save Version (commit) this notebook, then on the Output tab click')
            print('   "New Dataset" to publish gpr151_pipeline/. Attach that dataset as input')
            print('   on the next session (SECTION 0.5) to continue this run.')

    import glob, MDAnalysis as mda
    from MDAnalysis import Writer

    chunk_files = sorted(glob.glob(f'{WORK_DIR}/chunk_*.dcd'))
    print(f'Chunks found: {len(chunk_files)}')
    total_ns = 0
    for cf in chunk_files:
        n_frames = len(mda.Universe(f'{WORK_DIR}/gpr151_npt.pdb', cf).trajectory)
        total_ns += n_frames * 0.1  # 100 ps per frame
        print(f'  {os.path.basename(cf)}: {n_frames} frames ({n_frames*0.1:.1f} ns)')

    print(f'Total: {total_ns:.1f} ns across {len(chunk_files)} chunks')

    out_dcd = f'{WORK_DIR}/gpr151_production.dcd'

    if len(chunk_files) > 1:
        print('\nConcatenating chunks...')
        ref = mda.Universe(f'{WORK_DIR}/gpr151_npt.pdb', chunk_files[0])
        with Writer(out_dcd, ref.atoms.n_atoms) as W:
            for chunk_path in chunk_files:
                u = mda.Universe(f'{WORK_DIR}/gpr151_npt.pdb', chunk_path)
                for ts in u.trajectory:
                    W.write(u.atoms)
        n_total = len(mda.Universe(f'{WORK_DIR}/gpr151_npt.pdb', out_dcd).trajectory)
        print(f'✅ Combined trajectory: {n_total} frames -> gpr151_production.dcd')
    elif len(chunk_files) == 1:
        shutil.copy(chunk_files[0], out_dcd)
        print('Single chunk -> gpr151_production.dcd')
    else:
        print('⚠️  No chunks found')

    print('\nReady for Section 9 (pocket analysis).')




In [ ]:
def section_S4():  # Analysis + Docking + Screening (cells 34-52)
    # ## SECTION 9: Orthosteric Pocket Analysis
    # Analyze the MD trajectory to characterize the orthosteric pocket: volume, accessibility, and geometry. Identify the top-N snapshots with the most open/accessible binding site for ensemble docking.

    import MDAnalysis as mda
    from MDAnalysis.analysis import rms, contacts
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.spatial import ConvexHull
    
    # Load trajectory
    print('Loading MD trajectory...')
    u = mda.Universe('gpr151_pipeline/gpr151_npt.pdb',
                     'gpr151_pipeline/gpr151_production.dcd')
    
    print(f'Trajectory: {len(u.trajectory)} frames')
    print(f'Atoms: {len(u.atoms)}')
    
    # Select protein atoms only
    protein = u.select_atoms('protein')
    print(f'Protein atoms: {len(protein)}')
    
    # Select backbone for RMSD alignment
    backbone = u.select_atoms('protein and backbone')
    
    # --- RMSD analysis ---
    R = rms.RMSD(protein, backbone, groupselections=['protein and backbone'])
    R.run()
    
    rmsd_data = R.results.rmsd
    times = rmsd_data[:, 1]    # time in ps
    rmsd_vals = rmsd_data[:, 2]  # RMSD in Angstroms
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    axes[0].plot(times/1000, rmsd_vals, color='navy', lw=0.8)
    axes[0].set_xlabel('Time (ns)')
    axes[0].set_ylabel('RMSD (Å)')
    axes[0].set_title('GPR151-ΔN backbone RMSD over production MD')
    axes[0].axhline(y=np.mean(rmsd_vals), color='red', ls='--', lw=1, label=f'Mean: {np.mean(rmsd_vals):.2f} Å')
    axes[0].legend()
    
    # --- Estimate orthosteric pocket volume ---
    # The orthosteric pocket of Class A GPCRs is between TM3, TM5, TM6, TM7
    # Select residues in the canonical binding pocket region
    # Using GPCRDB generic positions 3.32, 3.33, 3.36, 5.42, 5.43, 6.48, 6.51, 6.52, 7.39, 7.42, 7.43
    # For now, select all residues within 10 Å of the pocket centroid
    
    # Define pocket centroid (center of TM bundle, extracellular half)
    # We'll approximate this as the centroid of TM3/5/6/7 Cα atoms in the extracellular region
    # This will be refined after the first frame analysis
    
    pocket_volumes = []
    frame_indices = []
    
    print('\nAnalyzing pocket volume across trajectory...')
    # Sample every 5th frame for speed
    for ts in u.trajectory[::5]:
        # Select protein Cα atoms roughly in the extracellular-facing TM region
        # (upper 1/3 of the 7TM bundle)
        ca_atoms = protein.select_atoms('name CA')
        ca_pos = ca_atoms.positions
    
        # Get z-range of protein, focus on upper extracellular third
        z_min, z_max = ca_pos[:, 2].min(), ca_pos[:, 2].max()
        z_ec_cutoff = z_min + (z_max - z_min) * 0.5  # upper half
    
        # Select atoms in pocket region (upper half of TM bundle)
        pocket_region = protein.select_atoms(
            f'(name CA or name CB or name CG) and prop z > {z_ec_cutoff}')
    
        if len(pocket_region) > 4:
            try:
                hull = ConvexHull(pocket_region.positions)
                pocket_volumes.append(hull.volume)  # in Å³
            except Exception:
                pocket_volumes.append(0)
        else:
            pocket_volumes.append(0)
        frame_indices.append(ts.frame)
    
    pocket_volumes = np.array(pocket_volumes)
    frame_indices = np.array(frame_indices)
    
    axes[1].plot(np.array(frame_indices) * 0.1, pocket_volumes, color='darkorange', lw=0.8)
    axes[1].set_xlabel('Time (ns)')
    axes[1].set_ylabel('Approximate pocket volume (Å³)')
    axes[1].set_title('Orthosteric pocket accessibility over MD trajectory')
    
    plt.tight_layout()
    plt.savefig('gpr151_pipeline/md_analysis.png', dpi=150)
    plt.show()
    
    print(f'Mean pocket volume: {np.mean(pocket_volumes):.0f} Å³')
    print(f'Max pocket volume: {np.max(pocket_volumes):.0f} Å³ at frame {frame_indices[np.argmax(pocket_volumes)]}')

    # Extract top-20 most open frames for ensemble docking
    N_DOCKING_FRAMES = 20
    
    # Only use frames from the second half of the trajectory (equilibrated)
    n_total = len(frame_indices)
    equilibrated_mask = frame_indices >= frame_indices[n_total // 2]
    
    eq_volumes = pocket_volumes[equilibrated_mask]
    eq_frames = frame_indices[equilibrated_mask]
    
    # Select top-N frames by pocket volume, ensuring spread across trajectory
    sorted_idx = np.argsort(eq_volumes)[::-1]
    selected_frames = []
    MIN_FRAME_SEPARATION = max(1, len(eq_frames) // (N_DOCKING_FRAMES * 2))
    
    for idx in sorted_idx:
        frame = eq_frames[idx]
        if all(abs(frame - sf) >= MIN_FRAME_SEPARATION for sf in selected_frames):
            selected_frames.append(frame)
        if len(selected_frames) >= N_DOCKING_FRAMES:
            break
    
    print(f'Selected {len(selected_frames)} frames for ensemble docking:')
    print(f'Frames: {sorted(selected_frames)}')
    
    # Export selected frames as PDB files for docking
    os.makedirs('gpr151_pipeline/docking_frames', exist_ok=True)
    
    for frame_num in selected_frames:
        u.trajectory[frame_num]
        protein_only = u.select_atoms('protein')
        with mda.Writer(f'gpr151_pipeline/docking_frames/frame_{frame_num:04d}.pdb',
                        protein_only.n_atoms) as W:
            W.write(protein_only)
    
    print(f'✅ Exported {len(selected_frames)} receptor conformations for ensemble docking')

    # ## SECTION 10: AutoDock Vina — Docking GUM-Class Psoralen Compounds
    # Dock psoralen scaffold and GUM-class analogs into the ensemble of open receptor conformations. The psoralen SMILES are the parent scaffold and several substituted analogs — replace with exact GUM3/GUM4 SMILES from Yoshida et al. 2026 (Genes to Cells, DOI: 10.1111/gtc.70119).

    from rdkit import Chem
    from rdkit.Chem import AllChem, Descriptors, Draw
    from rdkit.Chem.Draw import IPythonConsole
    import subprocess
    
    # GUM-class psoralen compound library
    # NOTE: Replace GUM3_SMILES and GUM4_SMILES with actual SMILES from Yoshida 2026
    # All have psoralen ring system (6H-furo[3,2-g][1]benzopyran-6-one)
    GUM_LIBRARY = {
        'Psoralen':          'O=c1ccc2cc3ccoc3cc2o1',
        '8-MOP (Methoxsalen)': 'COc1c2ccoc2cc2oc(=O)ccc12',
        '5-MOP (Bergapten)': 'COc1ccc2ccc(=O)oc2c1-c1ccoc1',
        'Isopsoralen':       'O=c1ccc2cc3occc3cc2o1',
        'Xanthotoxol':       'OCc1ccoc1-c1cc2ccc(=O)oc2cc1O',
        # Add GUM3/GUM4 SMILES from Yoshida et al. 2026 when obtained:
        # 'GUM3': '<SMILES from paper>',
        # 'GUM4': '<SMILES from paper>',
    }
    
    print('GUM-class compound library:')
    valid_compounds = {}
    for name, smiles in GUM_LIBRARY.items():
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            mw = Descriptors.ExactMolWt(mol)
            logp = Descriptors.MolLogP(mol)
            tpsa = Descriptors.TPSA(mol)
            hbd = Descriptors.NumHDonors(mol)
            hba = Descriptors.NumHAcceptors(mol)
            print(f'  {name}: MW={mw:.1f}, LogP={logp:.2f}, TPSA={tpsa:.1f}, HBD={hbd}, HBA={hba}')
            valid_compounds[name] = (smiles, mol)
        else:
            print(f'  {name}: INVALID SMILES')
    
    print(f'\n{len(valid_compounds)} valid compounds for docking')

    import subprocess
    import os
    
    os.makedirs('gpr151_pipeline/ligands', exist_ok=True)
    os.makedirs('gpr151_pipeline/docking_results', exist_ok=True)
    
    def prepare_ligand(name, smiles, output_dir):
        """Generate 3D conformation and convert to PDBQT for AutoDock Vina."""
        mol = Chem.MolFromSmiles(smiles)
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
        AllChem.MMFFOptimizeMolecule(mol)
    
        # Save as SDF
        sdf_path = f'{output_dir}/{name.replace(" ","_")}.sdf'
        w = Chem.SDWriter(sdf_path)
        w.write(mol)
        w.close()
    
        # Convert SDF → PDBQT using Open Babel
        pdbqt_path = f'{output_dir}/{name.replace(" ","_")}.pdbqt'
        result = subprocess.run(
            ['obabel', sdf_path, '-O', pdbqt_path, '-h', '--partialcharge', 'gasteiger'],
            capture_output=True, text=True)
        return pdbqt_path if os.path.exists(pdbqt_path) else None
    
    def prepare_receptor(pdb_path, output_path):
        """Convert receptor PDB to PDBQT for AutoDock Vina."""
        result = subprocess.run(
            ['obabel', pdb_path, '-O', output_path, '-xr'],
            capture_output=True, text=True)
        return output_path if os.path.exists(output_path) else None
    
    print('Preparing ligands...')
    ligand_pdbqts = {}
    for name, (smiles, mol) in valid_compounds.items():
        path = prepare_ligand(name, smiles, 'gpr151_pipeline/ligands')
        if path:
            ligand_pdbqts[name] = path
            print(f'  ✓ {name}')
        else:
            print(f'  ✗ {name} — preparation failed')
    
    print(f'\n{len(ligand_pdbqts)} ligands prepared')

    # Determine docking box center from pocket analysis
    # Use the frame with the largest pocket volume
    best_frame = frame_indices[np.argmax(pocket_volumes)]
    u.trajectory[best_frame]
    
    # Compute pocket centroid as center of TM bundle upper region
    protein_atoms = u.select_atoms('protein')
    ca_pos = u.select_atoms('protein and name CA').positions
    z_min, z_max = ca_pos[:, 2].min(), ca_pos[:, 2].max()
    z_upper = z_min + (z_max - z_min) * 0.55  # slightly above center
    
    pocket_atoms = u.select_atoms(f'protein and name CA and prop z > {z_upper}')
    pocket_center = pocket_atoms.positions.mean(axis=0)
    
    BOX_CENTER = tuple(pocket_center)  # (x, y, z) in Angstroms
    BOX_SIZE = (25.0, 25.0, 25.0)  # 25×25×25 Å box
    
    print(f'Docking box center: ({BOX_CENTER[0]:.2f}, {BOX_CENTER[1]:.2f}, {BOX_CENTER[2]:.2f}) Å')
    print(f'Docking box size: {BOX_SIZE} Å')

    from vina import Vina
    import json
    
    all_docking_results = {}
    
    # Ensemble docking: dock each ligand into each selected receptor conformation
    print(f'Running ensemble docking: {len(ligand_pdbqts)} ligands × {len(selected_frames)} receptor frames')
    print(f'Total docking runs: {len(ligand_pdbqts) * len(selected_frames)}')
    
    for lig_name, lig_path in ligand_pdbqts.items():
        scores_for_ligand = []
        print(f'\nDocking {lig_name}...')
    
        for frame_num in sorted(selected_frames)[:10]:  # First 10 frames for demo
            rec_pdb = f'gpr151_pipeline/docking_frames/frame_{frame_num:04d}.pdb'
            rec_pdbqt = rec_pdb.replace('.pdb', '.pdbqt')
    
            # Prepare receptor PDBQT
            if not os.path.exists(rec_pdbqt):
                prepare_receptor(rec_pdb, rec_pdbqt)
    
            if not os.path.exists(rec_pdbqt):
                continue
    
            try:
                v = Vina(sf_name='vina', verbosity=0)
                v.set_receptor(rec_pdbqt)
                v.set_ligand_from_file(lig_path)
                v.compute_vina_maps(
                    center=list(BOX_CENTER),
                    box_size=list(BOX_SIZE)
                )
                v.dock(exhaustiveness=16, n_poses=9)
                energies = v.energies()
                best_score = energies[0][0]  # Best pose affinity (kcal/mol)
                scores_for_ligand.append(best_score)
    
                # Save best pose
                out_path = f'gpr151_pipeline/docking_results/{lig_name.replace(" ","_")}_frame{frame_num:04d}.pdbqt'
                v.write_poses(out_path, n_poses=3, overwrite=True)
    
            except Exception as e:
                print(f'  Frame {frame_num}: docking failed — {e}')
    
        if scores_for_ligand:
            mean_score = np.mean(scores_for_ligand)
            best_score = min(scores_for_ligand)
            all_docking_results[lig_name] = {
                'mean_score': mean_score,
                'best_score': best_score,
                'all_scores': scores_for_ligand
            }
            print(f'  {lig_name}: best={best_score:.2f}, mean={mean_score:.2f} kcal/mol')
    
    # Save results
    with open('gpr151_pipeline/docking_results/ensemble_docking_summary.json', 'w') as f:
        json.dump(all_docking_results, f, indent=2)
    
    print('\n✅ Ensemble docking complete')

    # Visualize docking results
    import matplotlib.pyplot as plt
    import pandas as pd
    
    if all_docking_results:
        df_dock = pd.DataFrame([
            {'Compound': name, 'Best Score (kcal/mol)': data['best_score'],
             'Mean Score (kcal/mol)': data['mean_score']}
            for name, data in all_docking_results.items()
        ]).sort_values('Best Score (kcal/mol)')
    
        fig, ax = plt.subplots(figsize=(10, 5))
        x = range(len(df_dock))
        bars = ax.bar(x, df_dock['Best Score (kcal/mol)'], color='steelblue', label='Best pose')
        ax.scatter(x, df_dock['Mean Score (kcal/mol)'], color='red', zorder=5, label='Mean (ensemble)')
        ax.set_xticks(list(x))
        ax.set_xticklabels(df_dock['Compound'], rotation=30, ha='right')
        ax.set_ylabel('Docking score (kcal/mol)')
        ax.set_title('GPR151 Orthosteric Site — GUM-Class Ensemble Docking Results')
        ax.axhline(y=-6.0, color='orange', ls='--', lw=1, label='Typical hit threshold (-6 kcal/mol)')
        ax.legend()
        plt.tight_layout()
        plt.savefig('gpr151_pipeline/docking_scores.png', dpi=150)
        plt.show()
    
        print('\nDocking Results Summary:')
        print(df_dock.to_string(index=False))

    # ## SECTION 11: Key Residue Contact Analysis
    # Identify the GPR151 residues most frequently contacted by GUM-class compounds across the docking ensemble. These are the pharmacophoric hotspots for lead optimization and for the patent claim on key binding residues.

    import prolif as plf
    import MDAnalysis as mda
    from rdkit import Chem
    import pandas as pd
    import matplotlib.pyplot as plt
    import glob
    
    # Analyze protein-ligand interactions using ProLIF
    # For each ligand's best docking pose, compute interaction fingerprint
    
    print('Computing protein-ligand interaction fingerprints (ProLIF)...')
    
    all_interactions = []
    
    for lig_name, lig_path in ligand_pdbqts.items():
        # Find best docking pose file
        pose_files = glob.glob(f'gpr151_pipeline/docking_results/{lig_name.replace(" ","_")}_*.pdbqt')
        if not pose_files:
            continue
    
        # Use the receptor frame that gave the best score
        if lig_name in all_docking_results:
            best_score = all_docking_results[lig_name]['best_score']
            best_frame = selected_frames[np.argmin([
                all_docking_results[lig_name]['all_scores'][i]
                for i, _ in enumerate(sorted(selected_frames)[:10])
            ])]
        else:
            best_frame = sorted(selected_frames)[0]
    
        rec_pdb = f'gpr151_pipeline/docking_frames/frame_{best_frame:04d}.pdb'
        best_pose_file = f'gpr151_pipeline/docking_results/{lig_name.replace(" ","_")}_frame{best_frame:04d}.pdbqt'
    
        if not os.path.exists(best_pose_file) or not os.path.exists(rec_pdb):
            continue
    
        try:
            # Load with MDAnalysis
            u_rec = mda.Universe(rec_pdb)
            u_lig = mda.Universe(best_pose_file)
    
            rec_plf = plf.Molecule.from_mda(u_rec.select_atoms('protein'))
    
            # Convert ligand
            lig_mol = Chem.MolFromPDBBlock(u_lig.atoms.convert_to('RDKIT'), removeHs=False)
            if lig_mol is None:
                continue
            lig_plf = plf.Molecule(lig_mol)
    
            fp = plf.Fingerprint()
            fp.run_from_iterable([[lig_plf]], rec_plf)
            df_fp = fp.to_dataframe()
    
            # Extract residues with interactions
            for col in df_fp.columns:
                if df_fp[col].any():
                    res_name = str(col[0])
                    int_type = col[1]
                    all_interactions.append({
                        'Ligand': lig_name,
                        'Residue': res_name,
                        'Interaction': int_type
                    })
        except Exception as e:
            print(f'  {lig_name}: ProLIF failed — {e}')
    
    if all_interactions:
        df_int = pd.DataFrame(all_interactions)
        print(f'\nInteractions found: {len(df_int)}')
    
        # Frequency of residue contacts across all ligands
        residue_freq = df_int.groupby('Residue').size().sort_values(ascending=False)
        print('\nTop contacted residues (across all GUM-class compounds):')
        for res, count in residue_freq.head(15).items():
            print(f'  {res}: {count} interactions')
    
        # Heatmap: residues × ligands × interaction types
        pivot = df_int.pivot_table(index='Residue', columns='Ligand', aggfunc='count', fill_value=0)
        top_residues = residue_freq.head(20).index
        pivot_top = pivot.loc[pivot.index.isin(top_residues)].fillna(0)
    
        fig, ax = plt.subplots(figsize=(12, 8))
        import seaborn as sns
        sns.heatmap(pivot_top.astype(float), ax=ax, cmap='YlOrRd',
                    annot=True, fmt='.0f', linewidths=0.5)
        ax.set_title('GPR151 Orthosteric Residue Contacts — GUM-Class Compounds')
        ax.set_xlabel('Compound')
        ax.set_ylabel('Residue')
        plt.tight_layout()
        plt.savefig('gpr151_pipeline/contact_heatmap.png', dpi=150)
        plt.show()
    
        df_int.to_csv('gpr151_pipeline/all_interactions.csv', index=False)
        residue_freq.to_csv('gpr151_pipeline/key_residues_ranked.csv', header=['frequency'])
        print('\n✅ Key residue analysis complete')
    else:
        print('No interactions computed — check docking output files')

    # ## SECTION 12: N-Terminus Displacement — Steered MD Setup
    # Set up a steered MD experiment to model how GUM-class compounds displace the autoinhibitory N-terminus. We apply a pulling force along the collective variable defined as the distance between the best-docked GUM pose and the N-terminal domain.

    from openmm import CustomCVForce, RMSDForce
    import numpy as np
    
    # For the steered MD, we need the full-length GPR151 (with N-terminus)
    # loaded with the best-docked GUM compound placed in the orthosteric site
    # This section sets up the system and defines the pulling coordinate
    
    print('Setting up Steered MD for N-terminal displacement analysis...')
    print()
    print('Strategy:')
    print('  1. Start from full-length GPR151 (with N-terminus intact)')
    print('  2. Place best GUM compound in orthosteric pocket (from docking)')
    print('  3. Apply harmonic biasing force pulling GUM into pocket')
    print('  4. Monitor N-terminus RMSD as GUM penetrates deeper')
    print('  5. Free energy estimate via Jarzynski equality')
    print()
    
    # CV1: GUM compound center of mass depth into pocket (Z-coordinate relative to pocket center)
    # CV2: N-terminal domain RMSD from autoinhibited reference conformation
    # Pulling CV1 → GUM deeper into pocket → expect CV2 (N-term RMSD) to increase
    
    # Define the steered MD parameters
    SMD_CONFIG = {
        'pulling_rate': 0.001,      # nm/ps = 1 m/s (fast for demonstration)
        'spring_constant': 1000.0,  # kJ/mol/nm²
        'total_time_ns': 5.0,       # total steered MD time
        'save_every_ps': 1.0,       # save trajectory every 1 ps
        'n_replicates': 3,          # for Jarzynski averaging
    }
    
    print('Steered MD configuration:')
    for k, v in SMD_CONFIG.items():
        print(f'  {k}: {v}')
    
    print()
    print('Note: This section requires the full-length GPR151 system.')
    print('The steered MD will pull GUM3/GUM4 into the pocket while')
    print('tracking N-terminal domain displacement.')
    print()
    print('For full metadynamics free energy landscape, use PLUMED')
    print('(integrated with GROMACS — see companion PLUMED input below)')
    
    # Write PLUMED input for metadynamics (for use with GROMACS)
    plumed_input = f"""
    # PLUMED input for GPR151 N-terminal displacement metadynamics
    # Collective variables:
    #   cv1: GUM ligand center of mass Z-coordinate (pocket penetration depth)
    #   cv2: N-terminal domain RMSD from autoinhibited reference
    
    WHOLEMOLECULES ENTITY0=1-{len(protein)} ENTITY1={len(protein)+1}-{len(protein)+50}
    
    # CV1: Ligand Z-position relative to orthosteric pocket center
    lig_com: COM ATOMS={len(protein)+1}-{len(protein)+50}
    pocket_ref: FIXEDATOM AT={BOX_CENTER[0]/10:.3f},{BOX_CENTER[1]/10:.3f},{BOX_CENTER[2]/10:.3f}
    cv1: DISTANCE ATOMS=lig_com,pocket_ref COMPONENTS
    
    # CV2: N-terminal RMSD (residues 1-{TM1_START-1} Cα atoms)
    nterm_ca: GROUP ATOMS=@CA:1-{TM1_START-1}
    cv2: RMSD REFERENCE=gpr151_nterm_ref.pdb TYPE=OPTIMAL ATOMS=nterm_ca
    
    # Well-tempered metadynamics
    METAD ...
      ARG=cv1.z,cv2
      SIGMA=0.05,0.1
      HEIGHT=1.2
      BIASFACTOR=10
      TEMP=310
      PACE=500
      LABEL=metad
      FILE=HILLS
      GRID_MIN=-2.0,0.0
      GRID_MAX=2.0,3.0
      GRID_BIN=200,300
    ... METAD
    
    PRINT ARG=cv1.z,cv2,metad.bias STRIDE=500 FILE=COLVAR
    """
    
    with open('gpr151_pipeline/plumed_metadynamics.dat', 'w') as f:
        f.write(plumed_input)
    print('✅ PLUMED metadynamics input saved: gpr151_pipeline/plumed_metadynamics.dat')

    # ## SECTION 13: CNS Drug-Like Property Screening
    # Filter docking hits for CNS drug-like properties required for habenula/spinal cord targeting.

    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors
    import pandas as pd
    
    def cns_mpo_score(mol):
        """
        CNS Multiparameter Optimization score (Wager et al. 2010).
        Scores 6 properties, each contributing 0–1, max total = 6.0
        Compounds with CNS MPO > 4.0 have high probability of CNS penetrance.
        """
        mw = Descriptors.ExactMolWt(mol)
        clogp = Descriptors.MolLogP(mol)
        tpsa = Descriptors.TPSA(mol)
        hbd = Descriptors.NumHDonors(mol)
        pka_basic = 8.0  # estimated (would use pKa predictor in production)
        plogd = clogp - 0.025 * (pka_basic - 7.4) if pka_basic > 7.4 else clogp
    
        # Desirability functions for each property
        def desirability(value, low_good, high_good, low_ok, high_ok):
            if low_good <= value <= high_good:
                return 1.0
            elif low_ok <= value < low_good:
                return (value - low_ok) / (low_good - low_ok)
            elif high_good < value <= high_ok:
                return (high_ok - value) / (high_ok - high_good)
            return 0.0
    
        scores = {
            'MW': desirability(mw, 200, 360, 100, 500),
            'cLogP': desirability(clogp, 1, 3, -1, 5),
            'TPSA': desirability(tpsa, 20, 60, 0, 120),
            'HBD': desirability(hbd, 0, 1, 0, 3),
            'pKa_basic': desirability(pka_basic, 7.5, 9.5, 6, 11),
            'plogD': desirability(plogd, 1, 3, -1, 5),
        }
        total = sum(scores.values())
        return total, scores
    
    print('CNS Drug-Like Property Analysis — GUM-Class Compounds')
    print('='*65)
    
    results = []
    for name, (smiles, mol) in valid_compounds.items():
        mpo, scores = cns_mpo_score(mol)
        dock_score = all_docking_results.get(name, {}).get('best_score', 'N/A')
    
        results.append({
            'Compound': name,
            'MW': f"{Descriptors.ExactMolWt(mol):.1f}",
            'cLogP': f"{Descriptors.MolLogP(mol):.2f}",
            'TPSA': f"{Descriptors.TPSA(mol):.1f}",
            'HBD': Descriptors.NumHDonors(mol),
            'CNS_MPO': f"{mpo:.2f}/6.0",
            'Docking': f"{dock_score:.2f}" if isinstance(dock_score, float) else dock_score,
            'CNS_Ready': '✅' if mpo >= 4.0 else '⚠️' if mpo >= 3.0 else '❌'
        })
    
    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))
    
    df_results.to_csv('gpr151_pipeline/compound_ranking.csv', index=False)
    print('\n✅ Compound ranking saved: gpr151_pipeline/compound_ranking.csv')
    print('\nThreshold: CNS MPO ≥ 4.0 for CNS drug-like profile (required for habenula targeting)')

    # ## SECTION 14: Results Summary & Visualization
    # Compile all pipeline outputs into a final results summary.

    import py3Dmol
    from IPython.display import display, HTML
    
    # 3D visualization of the best-docked GUM compound in GPR151 orthosteric site
    # Load the receptor and best pose
    
    if all_docking_results:
        best_compound = min(all_docking_results, key=lambda x: all_docking_results[x]['best_score'])
        best_score_val = all_docking_results[best_compound]['best_score']
        best_frame_for_viz = sorted(selected_frames)[0]
    
        rec_pdb_path = f'gpr151_pipeline/docking_frames/frame_{best_frame_for_viz:04d}.pdb'
        lig_pdbqt_path = f'gpr151_pipeline/docking_results/{best_compound.replace(" ","_")}_frame{best_frame_for_viz:04d}.pdbqt'
    
        if os.path.exists(rec_pdb_path):
            with open(rec_pdb_path) as f:
                rec_pdb_str = f.read()
    
            view = py3Dmol.view(width=800, height=500)
            view.addModel(rec_pdb_str, 'pdb')
            view.setStyle({'chain': 'A'}, {
                'cartoon': {'color': 'spectrum', 'opacity': 0.8}
            })
    
            # Highlight orthosteric pocket region
            view.addSurface(py3Dmol.SES, {
                'opacity': 0.2,
                'color': 'lightblue'
            }, {'chain': 'A'})
    
            if os.path.exists(lig_pdbqt_path):
                with open(lig_pdbqt_path) as f:
                    lig_str = f.read()
                view.addModel(lig_str, 'pdbqt')
                view.setStyle({'model': 1}, {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.15}})
    
            view.zoomTo({'chain': 'A'})
            view.show()
    
            print(f'\n🏆 Best compound: {best_compound}')
            print(f'   Docking score: {best_score_val:.2f} kcal/mol')
    else:
        print('No docking results to visualize')

    # Final summary
    print('='*70)
    print('GPR151 COMPUTATIONAL PIPELINE — RESULTS SUMMARY')
    print('='*70)
    print()
    print(f'Target: GPR151 (UniProt Q8TDV0) — Orphan GPCR')
    print(f'Structure: AlphaFold v4 (wild-type, no thermostabilizing mutations)')
    print(f'Truncation: Residues 1–{TM1_START-1} removed (autoinhibitory N-terminus)')
    print(f'Membrane: POPC:POPE:Cholesterol (40:40:20), 0.15M NaCl, 310 K')
    print(f'MD: {PROD_NS} ns production run, CHARMM36m force field, OpenMM/CUDA')
    print(f'Pocket snapshots for docking: {len(selected_frames)} conformations')
    print()
    
    if all_docking_results:
        print('DOCKING RESULTS (GUM-class psoralen compounds):')
        for name, data in sorted(all_docking_results.items(),
                                 key=lambda x: x[1]['best_score']):
            print(f'  {name:30s}: best={data["best_score"]:.2f}, mean={data["mean_score"]:.2f} kcal/mol')
    
    print()
    print('OUTPUT FILES:')
    for f in sorted(os.listdir('gpr151_pipeline')):
        path = f'gpr151_pipeline/{f}'
        if os.path.isfile(path):
            size_kb = os.path.getsize(path) / 1024
            print(f'  {f}: {size_kb:.1f} KB')
    _c
    print()
    print('NEXT STEPS:')
    print('  1. Add exact GUM3/GUM4 SMILES from Yoshida et al. 2026 to compound library')
    print('  2. Run full 500 ns MD (extend on HPC cluster — download checkpoint.chk)')
    print('  3. Run metadynamics with PLUMED for N-terminus displacement free energy')
    print('  4. Expand virtual screen to ZINC psoralen subset')
    print('  5. Submit top candidates for in vitro GTPγS binding assay')
    print('='*70)

    # Package results — Kaggle exposes everything under /kaggle/working in the notebook's
    # Output tab automatically once you commit (Save Version -> Save & Run All). No explicit
    # download call needed; zip it up here for convenience anyway.
    import zipfile, os
    
    zip_path = 'GPR151_pipeline_results.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk('gpr151_pipeline'):
            for file in files:
                filepath = os.path.join(root, file)
                zf.write(filepath)
    
    print(f'✅ Results packaged: {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)')
    print('Available from this notebook\'s Output tab after commit, or publish gpr151_pipeline/')
    print('as a Kaggle Dataset to keep it independent of this notebook version.')




In [ ]:
# ── KAP CONDUCTOR ──
# This executes all sections in order with auto-debug

run_section("S0", section_S0)
run_section("S1", section_S1)
run_section("S2", section_S2)
run_section("S3", section_S3)
run_section("S4", section_S4)

status = final_report()
print(f"\n🏁 Final status: {status}")

if is_dirty():
    print("\n⚠️  Fixes applied! Re-commit this notebook to test them.")

In [ ]:
import glob
fix_files = sorted(glob.glob('/kaggle/working/kap_fixes/S2_*.py'))
for f in fix_files:
    print(f"\n{'='*60}")
    print(f"FILE: {f}")
    print(f"{'='*60}")
    with open(f) as fh:
        print(fh.read()[:2000])